In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

26/03/09 21:24:11 WARN Utils: Your hostname, setsus-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.1.216 instead (on interface en0)
26/03/09 21:24:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/09 21:24:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/09 21:24:13 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
!curl -O https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 67.8M  100 67.8M    0     0  6530k      0  0:00:10  0:00:10 --:--:-- 6525k


Question 1

In [4]:
spark.version

'3.5.8'

In [5]:
df = spark.read.parquet("yellow_tripdata_2025-11.parquet")

In [6]:
df_partition = df.repartition(4)

In [7]:
df.write.parquet('yellow_trip/2025/11')

Question 2
(base) xuehanyin@setsus-MacBook-Pro pyspark-zoomcamp % ls -lh yellow_trip/2025/11     
total 170496
-rw-r--r--  1 xuehanyin  staff     0B Mar  9 21:32 _SUCCESS
-rw-r--r--  1 xuehanyin  staff    21M Mar  9 21:32 part-00000-58540f99-1b57-436b-931d-b16eed70b50e-c000.snappy.parquet
-rw-r--r--  1 xuehanyin  staff    21M Mar  9 21:32 part-00002-58540f99-1b57-436b-931d-b16eed70b50e-c000.snappy.parquet
-rw-r--r--  1 xuehanyin  staff    21M Mar  9 21:32 part-00004-58540f99-1b57-436b-931d-b16eed70b50e-c000.snappy.parquet
-rw-r--r--  1 xuehanyin  staff    20M Mar  9 21:32 part-00006-58540f99-1b57-436b-931d-b16eed70b50e-c000.snappy.parquet

In [8]:
df.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee',
 'cbd_congestion_fee']

In [10]:
from pyspark.sql import functions as F

Question 3

In [15]:
df.filter(F.to_date("tpep_pickup_datetime") == "2025-11-15").count()

162604

In [20]:
df.registerTempTable('yellow_data')

/Users/xuehanyin/DE_training/pyspark-zoomcamp/.venv/lib/python3.8/site-packages/pyspark/sql/dataframe.py:329: FutureWarning: Deprecated in 2.0, use createOrReplaceTempView instead.
  warnings.warn("Deprecated in 2.0, use createOrReplaceTempView instead.", FutureWarning)


Question 4: longest trip

In [26]:
spark.sql("""
SELECT
    tpep_pickup_datetime,
    tpep_dropoff_datetime,
    DATEDIFF(MINUTE, tpep_pickup_datetime, tpep_dropoff_datetime) / 60
FROM
    yellow_data
ORDER BY 3 DESC
LIMIT 10
""").show()

[Stage 25:=============================>                            (4 + 4) / 8]

+--------------------+---------------------+-------------------------------------------------------------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|(timestampdiff(MINUTE, tpep_pickup_datetime, tpep_dropoff_datetime) / 60)|
+--------------------+---------------------+-------------------------------------------------------------------------+
| 2025-11-26 20:22:12|  2025-11-30 15:01:00|                                                        90.63333333333334|
| 2025-11-27 04:22:41|  2025-11-30 09:19:35|                                                        76.93333333333334|
| 2025-11-03 10:42:55|  2025-11-06 14:55:45|                                                                     76.2|
| 2025-11-07 11:23:22|  2025-11-10 08:40:41|                                                        69.28333333333333|
| 2025-11-18 17:12:47|  2025-11-21 12:17:37|                                                        67.06666666666666|
| 2025-11-22 17:45:30|  2025-11-25 09:07:36|    

In [27]:
!curl -O https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 12331  100 12331    0     0  74471      0 --:--:-- --:--:-- --:--:-- 74733


In [29]:
import pandas as pd

In [30]:
taxi_zone_lookup = pd.read_csv('taxi_zone_lookup.csv')

In [32]:
taxi_zone_lookup_spark = spark.createDataFrame(taxi_zone_lookup)

In [33]:
taxi_zone_lookup_spark.createOrReplaceTempView("taxi_zone_lookup")

In [34]:
taxi_zone_lookup_spark.show(5)

[Stage 26:>                                                         (0 + 1) / 1]

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows



In [37]:
joined_df = df.join(
    taxi_zone_lookup_spark,
    df.PULocationID == taxi_zone_lookup_spark.LocationID,
    "left"
)

In [39]:
joined_df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------+---------+-----------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|LocationID|  Borough|             Zone|service_zone|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+----------+---------+------

Question 6: Least frequent pickup location zone 

In [42]:
joined_df.groupby('Zone').count().orderBy("count").show(5)

+--------------------+-----+
|                Zone|count|
+--------------------+-----+
|Governor's Island...|    1|
|Eltingville/Annad...|    1|
|       Arden Heights|    1|
|       Port Richmond|    3|
|       Rikers Island|    4|
+--------------------+-----+
only showing top 5 rows

